In [ ]:
!pip install -q gradio scikit-image torchmetrics

In [ ]:
import os, math, random, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs: {n_gpus}")
if torch.cuda.is_available():
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
IMAGE_SIZE      = 128
CHANNELS        = 3
TIMESTEPS       = 300
BETA_START      = 1e-4
BETA_END        = 0.02
SCHEDULE        = 'cosine'   # 'linear' or 'cosine'
BATCH_SIZE      = 16
EPOCHS          = 10
LR              = 2e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
DIM             = 64         # base UNet channels
DIM_MULTS       = (1, 2, 4)  # → 64, 128, 256
NUM_WORKERS     = 4
RESULTS_DIR     = './ddpm_results'
CKPT_DIR        = './ddpm_checkpoints'
SAMPLE_EVERY    = 5
SAVE_EVERY      = 10
NUM_SAMPLES     = 5

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Config: {IMAGE_SIZE}x{IMAGE_SIZE} | T={TIMESTEPS} | BS={BATCH_SIZE} | Epochs={EPOCHS}")

In [ ]:
def find_dataset():
    candidates = [
        '/kaggle/input/celebahq256images-only',
        '/kaggle/input/celeba-hq-256/data256x256',
        '/kaggle/input/ffhq-face-data-set/thumbnails128x128',
        '/kaggle/input/ffhq-face-data-set',
        '/kaggle/input/wikiart/wikiart',
        '/kaggle/input/wikiart',
    ]
    for p in candidates:
        if os.path.exists(p):
            imgs = (glob.glob(f'{p}/**/*.jpg', recursive=True) +
                    glob.glob(f'{p}/**/*.png', recursive=True) +
                    glob.glob(f'{p}/**/*.jpeg', recursive=True))
            if len(imgs) > 50:
                print(f"✅ Dataset: {p}  ({len(imgs)} images)")
                return imgs
    # Fallback: download tiny FFHQ thumbnails subset via torchvision
    print("⚠️  No Kaggle dataset found – using CIFAR-10 as stand-in.")
    import torchvision.datasets as D
    ds = D.CIFAR10(root='/tmp/cifar', download=True, train=True)
    paths = []
    out = '/tmp/cifar_imgs'
    os.makedirs(out, exist_ok=True)
    for i,(img,_) in enumerate(ds):
        if i >= 500: break
        p2 = f'{out}/{i:05d}.png'
        img.save(p2)
        paths.append(p2)
    print(f"   Saved {len(paths)} CIFAR images to {out}")
    return paths

image_files = find_dataset()
random.shuffle(image_files)
split = int(0.95 * len(image_files))
train_files, val_files = image_files[:split], image_files[split:]
print(f"Train: {len(train_files)}  Val: {len(val_files)}")

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, files, size=128, augment=True):
        self.files = files
        ops = [transforms.Resize((size, size))]
        if augment:
            ops += [transforms.RandomHorizontalFlip()]
        ops += [
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),   # → [-1, 1]
        ]
        self.tf = transforms.Compose(ops)

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.files[i]).convert('RGB'))
        except:
            return torch.randn(3, IMAGE_SIZE, IMAGE_SIZE)

train_ds = ImageDataset(train_files, IMAGE_SIZE, augment=True)
val_ds   = ImageDataset(val_files,   IMAGE_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Preview
batch = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(18, 3))
fig.suptitle('Sample Training Images', fontsize=13, fontweight='bold')
for ax, img in zip(axes, batch[:8]):
    ax.imshow((img.permute(1,2,0).numpy() * .5 + .5).clip(0,1))
    ax.axis('off')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/sample_images.png', dpi=100); plt.show()

In [ ]:
def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps)
    ac = torch.cos(((x / T) + s) / (1 + s) * math.pi / 2) ** 2
    ac = ac / ac[0]
    betas = 1 - (ac[1:] / ac[:-1])
    return betas.clamp(1e-5, 0.9999)

def linear_beta_schedule(T, b_start=1e-4, b_end=0.02):
    return torch.linspace(b_start, b_end, T)

class DiffusionScheduler:
    def __init__(self, T=300, schedule='cosine'):
        self.T = T
        betas = cosine_beta_schedule(T) if schedule=='cosine' else linear_beta_schedule(T)
        self.betas    = betas
        alphas        = 1.0 - betas
        self.ac       = torch.cumprod(alphas, 0)            # ᾱ_t
        self.ac_prev  = F.pad(self.ac[:-1], (1,0), value=1.)
        self.sqrt_ac      = self.ac.sqrt()
        self.sqrt_1m_ac   = (1 - self.ac).sqrt()
        self.sqrt_recip_a = (1./alphas).sqrt()
        self.post_var     = betas * (1 - self.ac_prev) / (1 - self.ac)

    def _get(self, a, t, shape):
        v = a.gather(-1, t.cpu()).to(t.device)
        return v.reshape(t.shape[0], *([1]*(len(shape)-1)))

    def q_sample(self, x0, t, noise=None):
        """Forward: x_t = sqrt(ᾱ_t)*x0 + sqrt(1-ᾱ_t)*ε"""
        if noise is None: noise = torch.randn_like(x0)
        return self._get(self.sqrt_ac, t, x0.shape)*x0 +                self._get(self.sqrt_1m_ac, t, x0.shape)*noise

    def p_losses(self, model, x0, t, noise=None):
        """MSE loss: ||ε - ε_θ(x_t, t)||²"""
        if noise is None: noise = torch.randn_like(x0)
        x_noisy = self.q_sample(x0, t, noise)
        pred    = model(x_noisy, t)
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def p_sample(self, model, x, t_idx):
        t  = torch.full((x.shape[0],), t_idx, device=x.device, dtype=torch.long)
        bt = self._get(self.betas, t, x.shape)
        s1 = self._get(self.sqrt_1m_ac, t, x.shape)
        sr = self._get(self.sqrt_recip_a, t, x.shape)
        mean = sr * (x - bt * model(x, t) / s1)
        if t_idx == 0: return mean
        pv   = self._get(self.post_var, t, x.shape)
        return mean + pv.sqrt() * torch.randn_like(x)

    @torch.no_grad()
    def sample(self, model, shape, return_all=False):
        dev = next(model.parameters()).device
        img = torch.randn(shape, device=dev)
        imgs = [img.cpu()]
        for i in tqdm(reversed(range(self.T)), desc='Sampling', total=self.T, leave=False):
            img = self.p_sample(model, img, i)
            if return_all and i % (self.T//10) == 0:
                imgs.append(img.cpu())
        return (img.cpu(), imgs) if return_all else img.cpu()

scheduler = DiffusionScheduler(T=TIMESTEPS, schedule=SCHEDULE)
print(f"Scheduler ready | T={TIMESTEPS} | schedule={SCHEDULE}")
print(f"β: [{scheduler.betas.min():.5f}, {scheduler.betas.max():.5f}]")
print(f"ᾱ: [{scheduler.ac.min():.5f}, {scheduler.ac.max():.5f}]")

In [ ]:
def show_forward_diffusion(x0, steps_to_show=None):
    """Visualize image at selected noising timesteps."""
    if steps_to_show is None:
        steps_to_show = [0, TIMESTEPS//5, 2*TIMESTEPS//5,
                         3*TIMESTEPS//5, 4*TIMESTEPS//5, TIMESTEPS-1]
    fig, axes = plt.subplots(1, len(steps_to_show), figsize=(3*len(steps_to_show), 3))
    fig.suptitle('Forward Diffusion: Progressively Noising an Image', fontsize=13, fontweight='bold')
    for ax, step in zip(axes, steps_to_show):
        t = torch.tensor([step])
        xt = scheduler.q_sample(x0.unsqueeze(0), t)[0]
        img = (xt.permute(1,2,0).numpy() * .5 + .5).clip(0,1)
        ax.imshow(img)
        ax.set_title(f't = {step}', fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/forward_diffusion.png', dpi=120, bbox_inches='tight')
    plt.show()

sample_x0 = next(iter(train_loader))[0]   # first image
show_forward_diffusion(sample_x0)
print("✅ Forward diffusion visualization saved.")

In [ ]:
# ── Sinusoidal Time Embedding ───────────────────────────────────────────────
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        emb  = math.log(10000) / (half - 1)
        emb  = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb  = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

class TimeEmbedding(nn.Module):
    """Sinusoidal → 2-layer MLP → time projection."""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim * 4),
        )
    def forward(self, t): return self.net(t)

# ── Residual Block ──────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, groups=8):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch))
        self.block1   = nn.Sequential(nn.GroupNorm(groups, in_ch),  nn.SiLU(), nn.Conv2d(in_ch,  out_ch, 3, padding=1))
        self.block2   = nn.Sequential(nn.GroupNorm(groups, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.block1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.block2(h)
        return h + self.shortcut(x)

# ── Self-Attention Block ────────────────────────────────────────────────────
class AttentionBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm  = nn.GroupNorm(8, ch)
        self.attn  = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H*W).transpose(1,2)
        h, _ = self.attn(h, h, h)
        return x + h.transpose(1,2).reshape(B, C, H, W)

print("✅ Building blocks defined: SinusoidalPosEmb, TimeEmbedding, ResBlock, AttentionBlock")

In [ ]:
# ── U-Net ───────────────────────────────────────────────────────────────────
class UNet(nn.Module):
    """
    Simplified U-Net for DDPM.
    Channels: 64 → 128 → 256 (down) → 256 → 128 → 64 (up)
    """
    def __init__(self, in_ch=3, base_dim=64, dim_mults=(1,2,4),
                 attn_res=16, time_emb_dim=None):
        super().__init__()
        if time_emb_dim is None:
            time_emb_dim = base_dim * 4
        dims   = [base_dim * m for m in dim_mults]   # [64, 128, 256]
        in_out = list(zip([base_dim]+dims[:-1], dims))

        # Time embedding
        self.time_emb = TimeEmbedding(base_dim)

        # Initial conv
        self.init_conv = nn.Conv2d(in_ch, base_dim, 7, padding=3)

        # Encoder
        self.downs = nn.ModuleList()
        self.down_sample = nn.ModuleList()
        for i, (d_in, d_out) in enumerate(in_out):
            self.downs.append(nn.ModuleList([
                ResBlock(d_in,  d_out, base_dim*4),
                ResBlock(d_out, d_out, base_dim*4),
                AttentionBlock(d_out) if (IMAGE_SIZE // (2**i)) <= attn_res else nn.Identity(),
            ]))
            self.down_sample.append(
                nn.Conv2d(d_out, d_out, 4, 2, 1) if i < len(in_out)-1 else nn.Identity()
            )

        # Bottleneck
        mid = dims[-1]
        self.mid_res1 = ResBlock(mid, mid, base_dim*4)
        self.mid_attn = AttentionBlock(mid)
        self.mid_res2 = ResBlock(mid, mid, base_dim*4)

        # Decoder
        self.ups = nn.ModuleList()
        self.up_sample = nn.ModuleList()
        for i, (d_in, d_out) in enumerate(reversed(in_out)):
            self.ups.append(nn.ModuleList([
                ResBlock(d_out*2, d_in, base_dim*4),
                ResBlock(d_in,   d_in, base_dim*4),
                AttentionBlock(d_in) if (IMAGE_SIZE // (2**(len(in_out)-i-1))) <= attn_res else nn.Identity(),
            ]))
            self.up_sample.append(
                nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                              nn.Conv2d(d_in, d_in, 3, padding=1))
                if i < len(in_out)-1 else nn.Identity()
            )

        # Final
        self.final = nn.Sequential(
            nn.GroupNorm(8, base_dim),
            nn.SiLU(),
            nn.Conv2d(base_dim, in_ch, 1),
        )

    def forward(self, x, t):
        t_emb = self.time_emb(t)          # (B, base_dim*4)
        x = self.init_conv(x)
        skips = []

        for (r1, r2, attn), ds in zip(self.downs, self.down_sample):
            x = r1(x, t_emb); x = r2(x, t_emb); x = attn(x)
            skips.append(x)
            x = ds(x)

        x = self.mid_res1(x, t_emb)
        x = self.mid_attn(x)
        x = self.mid_res2(x, t_emb)

        for (r1, r2, attn), us in zip(self.ups, self.up_sample):
            x = torch.cat([x, skips.pop()], dim=1)
            x = r1(x, t_emb); x = r2(x, t_emb); x = attn(x)
            x = us(x)

        return self.final(x)

model = UNet(in_ch=CHANNELS, base_dim=DIM, dim_mults=DIM_MULTS).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ U-Net ready | Parameters: {n_params/1e6:.2f}M")

# Multi-GPU support
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"   DataParallel on {n_gpus} GPUs")

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
lr_sched  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR/10)
scaler    = GradScaler()          # Mixed-precision

train_losses, val_losses = [], []

def save_ckpt(epoch, loss):
    path = f'{CKPT_DIR}/ddpm_epoch{epoch:03d}.pt'
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optim': optimizer.state_dict(), 'loss': loss}, path)
    return path

def load_ckpt(path):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optim'])
    print(f"Loaded checkpoint: {path}  (epoch {ck['epoch']}, loss {ck['loss']:.4f})")
    return ck['epoch']

print("✅ Optimizer: AdamW | LR:", LR, "| Scheduler: CosineAnnealingLR")
print("   Mixed-precision scaler ready.")

In [ ]:
def train_epoch(epoch):
    model.train()
    total = 0.0
    pbar  = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for batch in pbar:
        x0 = batch.to(device)
        t  = torch.randint(0, TIMESTEPS, (x0.shape[0],), device=device).long()
        optimizer.zero_grad()
        with autocast():
            loss = scheduler.p_losses(model, x0, t)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total / len(train_loader)

@torch.no_grad()
def val_epoch():
    model.eval()
    total = 0.0
    for batch in val_loader:
        x0 = batch.to(device)
        t  = torch.randint(0, TIMESTEPS, (x0.shape[0],), device=device).long()
        with autocast():
            loss = scheduler.p_losses(model, x0, t)
        total += loss.item()
    return total / max(len(val_loader), 1)

# ── Main training loop ──────────────────────────────────────────────────────
print("🚀 Starting training...")
for epoch in range(1, EPOCHS + 1):
    t_loss = train_epoch(epoch)
    v_loss = val_epoch()
    lr_sched.step()
    train_losses.append(t_loss)
    val_losses.append(v_loss)
    print(f"Epoch {epoch:3d}/{EPOCHS}  train={t_loss:.4f}  val={v_loss:.4f}  lr={lr_sched.get_last_lr()[0]:.2e}")
    if epoch % SAMPLE_EVERY == 0:
        model.eval()
        samples = scheduler.sample(model, (4, CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
        grid    = make_grid((samples*.5+.5).clamp(0,1), nrow=4)
        save_image(grid, f'{RESULTS_DIR}/samples_epoch{epoch:03d}.png')
    if epoch % SAVE_EVERY == 0:
        save_ckpt(epoch, t_loss)
print("✅ Training complete!")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(train_losses)+1), train_losses, label='Train Loss', color='#4f8ef7', lw=2)
ax.plot(range(1, len(val_losses)+1),   val_losses,   label='Val Loss',   color='#f74f4f', lw=2, ls='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('DDPM Training Loss Curve', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final train loss: {train_losses[-1]:.4f} | val loss: {val_losses[-1]:.4f}")


In [ ]:
# ── Part 6: Generate 5 images from pure noise ───────────────────────────────
model.eval()
print("🎨 Generating images from pure noise...")
generated = scheduler.sample(model, (NUM_SAMPLES, CHANNELS, IMAGE_SIZE, IMAGE_SIZE))

fig, axes = plt.subplots(1, NUM_SAMPLES, figsize=(4*NUM_SAMPLES, 4))
fig.suptitle('Generated Images (from Pure Noise)', fontsize=14, fontweight='bold')
for i, (ax, img) in enumerate(zip(axes, generated)):
    ax.imshow((img.permute(1,2,0).numpy()*.5+.5).clip(0,1))
    ax.set_title(f'Sample {i+1}'); ax.axis('off')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/generated_images.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ {NUM_SAMPLES} images generated and saved.")

In [ ]:
# ── Part 5: Reconstruction ──────────────────────────────────────────────────
# Pick a target image from the val set
target_img = val_ds[0].unsqueeze(0)  # (1, C, H, W) in [-1,1]

# Noise to ~80% of total timesteps (t = 0.8*T) then denoise
t_frac = int(0.8 * TIMESTEPS)
t_tensor = torch.tensor([t_frac])
noised = scheduler.q_sample(target_img, t_tensor)

# Denoise from that noisy state
model.eval()
x = noised.to(device)
with torch.no_grad():
    for i in tqdm(reversed(range(t_frac)), desc='Reconstructing', total=t_frac, leave=False):
        t_ = torch.full((1,), i, device=device, dtype=torch.long)
        bt  = scheduler._get(scheduler.betas, t_, x.shape)
        s1  = scheduler._get(scheduler.sqrt_1m_ac, t_, x.shape)
        sr  = scheduler._get(scheduler.sqrt_recip_a, t_, x.shape)
        with autocast():
            pred_noise = model(x, t_)
        mean = sr * (x - bt * pred_noise / s1)
        if i > 0:
            pv = scheduler._get(scheduler.post_var, t_, x.shape)
            x  = mean + pv.sqrt() * torch.randn_like(x)
        else:
            x  = mean
recon = x.cpu()

# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Image Reconstruction: Target vs Generated', fontsize=14, fontweight='bold')
for ax, img, title in zip(axes,
        [target_img[0], noised[0], recon[0]],
        ['Target Image', f'Noised (t={t_frac})', 'Reconstructed']):
    ax.imshow((img.permute(1,2,0).numpy()*.5+.5).clip(0,1))
    ax.set_title(title, fontsize=12); ax.axis('off')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Reconstruction complete and saved.")

In [ ]:
def full_visualization(model, scheduler, sample_img, n_steps=5):
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle('DDPM Visualization: Forward / Reverse / Generated', fontsize=16, fontweight='bold')
    gs  = fig.add_gridspec(3, n_steps, hspace=0.4, wspace=0.3)

    # ── Row 1: Forward diffusion ────────────────────────────────────────────
    fwd_steps = [min(int(i*TIMESTEPS/(n_steps-1)), TIMESTEPS-1) for i in range(n_steps)]
# → [0, 75, 150, 225, 299]  ✅
    for col, step in enumerate(fwd_steps):
        t  = torch.tensor([step])
        xt = scheduler.q_sample(sample_img.unsqueeze(0), t)[0]
        ax = fig.add_subplot(gs[0, col])
        ax.imshow((xt.permute(1,2,0).numpy()*.5+.5).clip(0,1))
        ax.set_title(f'Fwd t={step}', fontsize=9); ax.axis('off')

    # ── Row 2: Reverse diffusion (intermediates) ────────────────────────────
    model.eval()
    _, intermediates = scheduler.sample(
        model, (1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE), return_all=True)
    step_idx = np.linspace(0, len(intermediates)-1, n_steps, dtype=int)
    for col, idx in enumerate(step_idx):
        img = intermediates[idx][0]
        ax  = fig.add_subplot(gs[1, col])
        ax.imshow((img.permute(1,2,0).numpy()*.5+.5).clip(0,1))
        ax.set_title(f'Rev step {idx}', fontsize=9); ax.axis('off')

    # ── Row 3: 5 distinct generated images ─────────────────────────────────
    gen = scheduler.sample(model, (n_steps, CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
    for col in range(n_steps):
        ax = fig.add_subplot(gs[2, col])
        ax.imshow((gen[col].permute(1,2,0).numpy()*.5+.5).clip(0,1))
        ax.set_title(f'Generated {col+1}', fontsize=9); ax.axis('off')

    # Row labels
    for row, label in enumerate(['Forward Diffusion', 'Reverse Diffusion', 'New Images']):
        fig.text(0.01, 1 - (row*0.33+0.17), label, fontsize=11,
                 fontweight='bold', va='center', rotation=90)

    plt.savefig(f'{RESULTS_DIR}/full_visualization.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("✅ Full visualization saved.")

full_visualization(model, scheduler, val_ds[0])

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

def to_np(t):
    """Tensor [-1,1] → numpy [0,1] HWC."""
    return (t.permute(1,2,0).numpy() * .5 + .5).clip(0,1)

# Evaluate reconstruction quality on val set (up to 20 images)
model.eval()
psnr_scores, ssim_scores = [], []
eval_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

print("Computing PSNR & SSIM on validation set...")
for i, x0 in enumerate(tqdm(eval_loader, desc='Evaluating')):
    if i >= 20: break
    # Noise at t=0.5*T then reconstruct
    t_val = int(0.5 * TIMESTEPS)
    t_tensor = torch.tensor([t_val])
    x_noisy  = scheduler.q_sample(x0, t_tensor).to(device)
    x = x_noisy.clone()
    with torch.no_grad():
        for step in reversed(range(t_val)):
            t_ = torch.full((1,), step, device=device, dtype=torch.long)
            bt  = scheduler._get(scheduler.betas, t_, x.shape)
            s1  = scheduler._get(scheduler.sqrt_1m_ac, t_, x.shape)
            sr  = scheduler._get(scheduler.sqrt_recip_a, t_, x.shape)
            pred_noise = model(x, t_)
            mean = sr * (x - bt * pred_noise / s1)
            if step > 0:
                pv = scheduler._get(scheduler.post_var, t_, x.shape)
                x  = mean + pv.sqrt() * torch.randn_like(x)
            else:
                x  = mean
    orig  = to_np(x0[0])
    recon = to_np(x.cpu()[0])
    psnr_scores.append(psnr_fn(orig, recon, data_range=1.0))
    ssim_scores.append(ssim_fn(orig, recon, data_range=1.0, channel_axis=2))

print(f"\n{'='*40}")
print(f"📊 Quantitative Evaluation Results")
print(f"{'='*40}")
print(f"  PSNR  →  Mean: {np.mean(psnr_scores):.2f} dB  |  Std: {np.std(psnr_scores):.2f}")
print(f"  SSIM  →  Mean: {np.mean(ssim_scores):.4f}    |  Std: {np.std(ssim_scores):.4f}")
print(f"{'='*40}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(psnr_scores)), psnr_scores, color='#4f8ef7', alpha=0.8)
axes[0].axhline(np.mean(psnr_scores), color='red', ls='--', label=f'Mean={np.mean(psnr_scores):.2f}')
axes[0].set_title('PSNR per Image (dB)', fontweight='bold'); axes[0].legend(); axes[0].set_xlabel('Image Index')
axes[1].bar(range(len(ssim_scores)), ssim_scores, color='#f7a34f', alpha=0.8)
axes[1].axhline(np.mean(ssim_scores), color='red', ls='--', label=f'Mean={np.mean(ssim_scores):.4f}')
axes[1].set_title('SSIM per Image', fontweight='bold'); axes[1].legend(); axes[1].set_xlabel('Image Index')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/psnr_ssim.png', dpi=130, bbox_inches='tight')
plt.show()